In [1]:
import pandas as pd

# ============================================================
# 1. LOAD DATA
# ============================================================

outpatient = pd.read_csv(
    "../data/processed/primary/outpatient_claim_features_for_combining.csv",
    low_memory=False
)

beneficiary = pd.read_csv(
    "../data/processed/primary/beneficiary_longitudinal.csv",
    low_memory=False
)

print("Outpatient:", outpatient.shape)
print("Beneficiary:", beneficiary.shape)


# ============================================================
# 2. CHECK BENEFICIARY FILE STRUCTURE
# ============================================================

print("\nBeneficiary columns:")
print(beneficiary.columns.tolist())

print("\nBeneficiary missing values:")
print(beneficiary.isna().sum())

print("\nUnique beneficiaries:")
print(beneficiary["DESYNPUF_ID"].nunique())


# ============================================================
# 3. IDENTIFY THE YEAR COLUMN
# ============================================================

year_candidates = [
    c for c in beneficiary.columns
    if c.upper() in ["YEAR", "BENEFICIARY_YEAR", "CLAIM_YEAR"]
]

print("\nPossible year columns:", year_candidates)

if len(year_candidates) == 0:
    print("\nWARNING: No obvious beneficiary year column found.")
else:
    beneficiary_year_col = year_candidates[0]
    print("Using beneficiary year column:", beneficiary_year_col)


# ============================================================
# 4. CREATE CLAIM YEAR CHECK
# ============================================================

print("\nOutpatient claim years:")
print(
    outpatient["CLAIM_YEAR"]
    .value_counts(dropna=False)
    .sort_index()
)


# ============================================================
# 5. CHECK BENEFICIARY + YEAR UNIQUENESS
# ============================================================

if len(year_candidates) > 0:

    duplicate_beneficiary_year = beneficiary.duplicated(
        subset=["DESYNPUF_ID", beneficiary_year_col],
        keep=False
    )

    print("\nDuplicate DESYNPUF_ID + YEAR rows:")
    print(duplicate_beneficiary_year.sum())

    if duplicate_beneficiary_year.sum() > 0:
        print("\nExample duplicates:")
        print(
            beneficiary.loc[
                duplicate_beneficiary_year,
                ["DESYNPUF_ID", beneficiary_year_col]
            ]
            .sort_values(["DESYNPUF_ID", beneficiary_year_col])
            .head(20)
        )


# ============================================================
# 6. PREPARE JOIN KEYS
# ============================================================

if len(year_candidates) > 0:

    beneficiary["_JOIN_YEAR"] = pd.to_numeric(
        beneficiary[beneficiary_year_col],
        errors="coerce"
    )

    outpatient["_JOIN_YEAR"] = pd.to_numeric(
        outpatient["CLAIM_YEAR"],
        errors="coerce"
    )

    print("\nBeneficiary years:")
    print(
        beneficiary["_JOIN_YEAR"]
        .value_counts(dropna=False)
        .sort_index()
    )


# ============================================================
# 7. CHECK BENEFICIARY ID OVERLAP
# ============================================================

outpatient_beneficiaries = set(
    outpatient["DESYNPUF_ID"].dropna().unique()
)

beneficiary_ids = set(
    beneficiary["DESYNPUF_ID"].dropna().unique()
)

matched_ids = outpatient_beneficiaries.intersection(
    beneficiary_ids
)

unmatched_ids = outpatient_beneficiaries.difference(
    beneficiary_ids
)

print("\n====================================")
print("BENEFICIARY ID OVERLAP")
print("====================================")

print("Outpatient unique beneficiaries:",
      len(outpatient_beneficiaries))

print("Beneficiary unique IDs:",
      len(beneficiary_ids))

print("Matched beneficiary IDs:",
      len(matched_ids))

print("Unmatched outpatient beneficiary IDs:",
      len(unmatched_ids))


# ============================================================
# 8. CHECK CLAIM-LEVEL YEAR MATCHING
# ============================================================

if len(year_candidates) > 0:

    beneficiary_keys = set(
        zip(
            beneficiary["DESYNPUF_ID"],
            beneficiary["_JOIN_YEAR"]
        )
    )

    outpatient_keys = list(
        zip(
            outpatient["DESYNPUF_ID"],
            outpatient["_JOIN_YEAR"]
        )
    )

    matched_rows = sum(
        key in beneficiary_keys
        for key in outpatient_keys
    )

    unmatched_rows = len(outpatient) - matched_rows

    print("\n====================================")
    print("BENEFICIARY-YEAR MATCHING")
    print("====================================")

    print("Total outpatient rows:", len(outpatient))
    print("Matched beneficiary-year rows:", matched_rows)
    print("Unmatched beneficiary-year rows:", unmatched_rows)

    print(
        "Match rate:",
        round(matched_rows / len(outpatient) * 100, 4),
        "%"
    )


# ============================================================
# 9. CHECK FOR MANY-TO-MANY JOIN RISK
# ============================================================

if len(year_candidates) > 0:

    beneficiary_key_counts = (
        beneficiary
        .groupby(
            ["DESYNPUF_ID", "_JOIN_YEAR"],
            dropna=False
        )
        .size()
    )

    print("\n====================================")
    print("MANY-TO-MANY JOIN CHECK")
    print("====================================")

    print(
        "Maximum beneficiary rows per ID-year:",
        beneficiary_key_counts.max()
    )

    print(
        "ID-year keys appearing more than once:",
        (beneficiary_key_counts > 1).sum()
    )

    print("\nBeneficiary rows per ID-year:")
    print(
        beneficiary_key_counts.describe()
    )


# ============================================================
# 10. CLEAN TEMPORARY COLUMNS
# ============================================================

if "_JOIN_YEAR" in outpatient.columns:
    outpatient.drop(columns="_JOIN_YEAR", inplace=True)

if "_JOIN_YEAR" in beneficiary.columns:
    beneficiary.drop(columns="_JOIN_YEAR", inplace=True)

Outpatient: (790790, 26)
Beneficiary: (343644, 33)

Beneficiary columns:
['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR', 'YEAR']

Beneficiary missing values:
DESYNPUF_ID                      0
BENE_BIRTH_DT                    0
BENE_DEATH_DT               338183
BENE_SEX_IDENT_CD                0
BENE_RACE_CD                     0
BENE_ESRD_IND                    0
SP_STATE_CODE                    0
BENE_COUNTY_CD                   0
BENE_HI_CVRAGE_TOT_MONS          0
BENE_SMI_CVRAGE_TOT_MONS         0
BENE_HMO_CVRAGE_TOT_MO

In [2]:
import pandas as pd

# Load
outpatient = pd.read_csv(
    "../data/processed/primary/outpatient_claim_features_for_combining.csv",
    low_memory=False
)

beneficiary = pd.read_csv(
    "../data/processed/primary/beneficiary_longitudinal.csv",
    low_memory=False
)

# Rename beneficiary YEAR for clarity
beneficiary = beneficiary.rename(columns={"YEAR": "CLAIM_YEAR"})

# Merge
outpatient_beneficiary = outpatient.merge(
    beneficiary,
    on=["DESYNPUF_ID", "CLAIM_YEAR"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_BENE")
)

print("====================================")
print("OUTPATIENT + BENEFICIARY JOIN")
print("====================================")

print("Original outpatient shape:", outpatient.shape)
print("Beneficiary shape:", beneficiary.shape)
print("Merged shape:", outpatient_beneficiary.shape)

print("\nOriginal rows:", len(outpatient))
print("Merged rows:", len(outpatient_beneficiary))

print("\nOriginal unique CLAIM_KEY:",
      outpatient["CLAIM_KEY"].nunique())

print("Merged unique CLAIM_KEY:",
      outpatient_beneficiary["CLAIM_KEY"].nunique())

print("\nRow count preserved:",
      len(outpatient) == len(outpatient_beneficiary))

print("\nClaim key preserved:",
      outpatient["CLAIM_KEY"].nunique()
      == outpatient_beneficiary["CLAIM_KEY"].nunique())

# How many claims received beneficiary information?
beneficiary_feature_cols = [
    c for c in beneficiary.columns
    if c not in ["DESYNPUF_ID", "CLAIM_YEAR"]
]

has_beneficiary_data = (
    outpatient_beneficiary[beneficiary_feature_cols]
    .notna()
    .any(axis=1)
)

print("\nClaims with beneficiary-year data:",
      has_beneficiary_data.sum())

print("Claims without beneficiary-year data:",
      (~has_beneficiary_data).sum())

print("\nMatch rate:",
      round(has_beneficiary_data.mean() * 100, 4),
      "%")

# Check duplicate claim keys after merge
print("\nDuplicate CLAIM_KEY after merge:",
      outpatient_beneficiary["CLAIM_KEY"].duplicated().sum())

# Save
output_path = (
    "../data/processed/primary/"
    "outpatient_claims_with_beneficiary_features.csv"
)

outpatient_beneficiary.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)
print("Final shape:", outpatient_beneficiary.shape)

OUTPATIENT + BENEFICIARY JOIN
Original outpatient shape: (790790, 26)
Beneficiary shape: (343644, 33)
Merged shape: (790790, 57)

Original rows: 790790
Merged rows: 790790

Original unique CLAIM_KEY: 790790
Merged unique CLAIM_KEY: 790790

Row count preserved: True

Claim key preserved: True

Claims with beneficiary-year data: 779225
Claims without beneficiary-year data: 11565

Match rate: 98.5375 %

Duplicate CLAIM_KEY after merge: 0

Saved: ../data/processed/primary/outpatient_claims_with_beneficiary_features.csv
Final shape: (790790, 57)


In [3]:
import pandas as pd

# Load processed inpatient claims
inpatient = pd.read_csv(
    "../data/processed/primary/inpatient_claims_cleaned.csv",
    low_memory=False
)

# Load beneficiary longitudinal data
beneficiary = pd.read_csv(
    "../data/processed/primary/beneficiary_longitudinal.csv",
    low_memory=False
)

print("====================================")
print("INPATIENT + BENEFICIARY PRE-CHECK")
print("====================================")

print("Inpatient shape:", inpatient.shape)
print("Beneficiary shape:", beneficiary.shape)

print("\nInpatient columns:")
print(inpatient.columns.tolist())

print("\nBeneficiary columns:")
print(beneficiary.columns.tolist())

# Identify inpatient year
if "CLAIM_YEAR" in inpatient.columns:
    inpatient_year_col = "CLAIM_YEAR"
else:
    inpatient["CLAIM_YEAR"] = pd.to_datetime(
        inpatient["CLM_FROM_DT"],
        errors="coerce"
    ).dt.year
    inpatient_year_col = "CLAIM_YEAR"

# Rename beneficiary year
beneficiary = beneficiary.rename(
    columns={"YEAR": "CLAIM_YEAR"}
)

# Make sure both join columns have compatible types
inpatient["CLAIM_YEAR"] = pd.to_numeric(
    inpatient["CLAIM_YEAR"],
    errors="coerce"
).astype("Int64")

beneficiary["CLAIM_YEAR"] = pd.to_numeric(
    beneficiary["CLAIM_YEAR"],
    errors="coerce"
).astype("Int64")

print("\nInpatient claim years:")
print(inpatient["CLAIM_YEAR"].value_counts(dropna=False).sort_index())

print("\nBeneficiary years:")
print(beneficiary["CLAIM_YEAR"].value_counts(dropna=False).sort_index())

# Beneficiary ID coverage
inpatient_ids = inpatient["DESYNPUF_ID"].nunique()
beneficiary_ids = beneficiary["DESYNPUF_ID"].nunique()

matched_ids = len(
    set(inpatient["DESYNPUF_ID"])
    & set(beneficiary["DESYNPUF_ID"])
)

print("\n====================================")
print("BENEFICIARY ID OVERLAP")
print("====================================")

print("Inpatient unique beneficiaries:", inpatient_ids)
print("Beneficiary unique IDs:", beneficiary_ids)
print("Matched beneficiary IDs:", matched_ids)
print(
    "Unmatched inpatient beneficiary IDs:",
    inpatient_ids - matched_ids
)

# Check beneficiary ID + year uniqueness
beneficiary_key_duplicates = beneficiary.duplicated(
    ["DESYNPUF_ID", "CLAIM_YEAR"]
).sum()

print("\n====================================")
print("BENEFICIARY-YEAR KEY CHECK")
print("====================================")

print(
    "Duplicate beneficiary ID + year rows:",
    beneficiary_key_duplicates
)

# Match rate BEFORE merge
beneficiary_keys = set(
    zip(
        beneficiary["DESYNPUF_ID"],
        beneficiary["CLAIM_YEAR"]
    )
)

inpatient_keys = list(
    zip(
        inpatient["DESYNPUF_ID"],
        inpatient["CLAIM_YEAR"]
    )
)

matched_rows = sum(
    key in beneficiary_keys
    for key in inpatient_keys
)

print("\nTotal inpatient rows:", len(inpatient))
print("Matched beneficiary-year rows:", matched_rows)
print(
    "Unmatched beneficiary-year rows:",
    len(inpatient) - matched_rows
)

print(
    "Match rate:",
    round(matched_rows / len(inpatient) * 100, 4),
    "%"
)

# Actual merge
inpatient_beneficiary = inpatient.merge(
    beneficiary,
    on=["DESYNPUF_ID", "CLAIM_YEAR"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_BENE")
)

print("\n====================================")
print("INPATIENT + BENEFICIARY JOIN")
print("====================================")

print("Original inpatient shape:", inpatient.shape)
print("Beneficiary shape:", beneficiary.shape)
print("Merged shape:", inpatient_beneficiary.shape)

print("\nOriginal rows:", len(inpatient))
print("Merged rows:", len(inpatient_beneficiary))

print(
    "\nRow count preserved:",
    len(inpatient) == len(inpatient_beneficiary)
)

# CLAIM_KEY validation
if "CLAIM_KEY" in inpatient.columns:
    print(
        "\nOriginal unique CLAIM_KEY:",
        inpatient["CLAIM_KEY"].nunique()
    )

    print(
        "Merged unique CLAIM_KEY:",
        inpatient_beneficiary["CLAIM_KEY"].nunique()
    )

    print(
        "Duplicate CLAIM_KEY after merge:",
        inpatient_beneficiary["CLAIM_KEY"].duplicated().sum()
    )

# Check beneficiary data actually attached
beneficiary_feature_cols = [
    c for c in beneficiary.columns
    if c not in ["DESYNPUF_ID", "CLAIM_YEAR"]
]

has_beneficiary_data = (
    inpatient_beneficiary[beneficiary_feature_cols]
    .notna()
    .any(axis=1)
)

print(
    "\nClaims with beneficiary-year data:",
    has_beneficiary_data.sum()
)

print(
    "Claims without beneficiary-year data:",
    (~has_beneficiary_data).sum()
)

print(
    "Actual match rate:",
    round(has_beneficiary_data.mean() * 100, 4),
    "%"
)

# Save
output_path = (
    "../data/processed/primary/"
    "inpatient_claims_with_beneficiary_features.csv"
)

inpatient_beneficiary.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)
print("Final shape:", inpatient_beneficiary.shape)

INPATIENT + BENEFICIARY PRE-CHECK
Inpatient shape: (66773, 42)
Beneficiary shape: (343644, 33)

Inpatient columns:
['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI', 'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'CLM_ADMSN_DT', 'ADMTNG_ICD9_DGNS_CD', 'CLM_PASS_THRU_PER_DIEM_AMT', 'NCH_BENE_IP_DDCTBL_AMT', 'NCH_BENE_PTA_COINSRNC_LBLTY_AM', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'CLM_UTLZTN_DAY_CNT', 'NCH_BENE_DSCHRG_DT', 'CLM_DRG_CD', 'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4', 'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8', 'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10', 'ICD9_PRCDR_CD_1', 'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4', 'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6', 'CLAIM_KEY', 'CLAIM_DURATION_DAYS', 'DIAGNOSIS_COUNT', 'PROCEDURE_COUNT', 'HAS_NEGATIVE_PAYMENT', 'HAS_PRIMARY_PAYER_PAYMENT']

Beneficiary columns:
['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT',

In [4]:
import pandas as pd

# Load enriched inpatient claims
inpatient = pd.read_csv(
    "../data/processed/primary/inpatient_claims_with_beneficiary_features.csv",
    low_memory=False
)

# Load provider-level features
provider = pd.read_csv(
    "../data/processed/primary/inpatient_provider_features.csv",
    low_memory=False
)

print("====================================")
print("INPATIENT + PROVIDER PRE-CHECK")
print("====================================")

print("Inpatient shape:", inpatient.shape)
print("Provider feature shape:", provider.shape)

print("\nInpatient providers:", inpatient["PRVDR_NUM"].nunique())
print("Provider feature records:", provider["PRVDR_NUM"].nunique())

# --------------------------------------------------
# PROVIDER KEY CHECK
# --------------------------------------------------

print("\n====================================")
print("PROVIDER KEY CHECK")
print("====================================")

provider_duplicate_keys = provider["PRVDR_NUM"].duplicated().sum()

print(
    "Duplicate PRVDR_NUM in provider features:",
    provider_duplicate_keys
)

print(
    "Unique provider feature keys:",
    provider["PRVDR_NUM"].nunique()
)

# --------------------------------------------------
# PROVIDER ID OVERLAP
# --------------------------------------------------

inpatient_providers = set(
    inpatient["PRVDR_NUM"].dropna()
)

feature_providers = set(
    provider["PRVDR_NUM"].dropna()
)

matched_providers = (
    inpatient_providers & feature_providers
)

unmatched_providers = (
    inpatient_providers - feature_providers
)

print("\n====================================")
print("PROVIDER ID OVERLAP")
print("====================================")

print(
    "Unique inpatient providers:",
    len(inpatient_providers)
)

print(
    "Provider feature IDs:",
    len(feature_providers)
)

print(
    "Matched providers:",
    len(matched_providers)
)

print(
    "Unmatched inpatient providers:",
    len(unmatched_providers)
)

print(
    "Provider match rate:",
    round(
        len(matched_providers)
        / len(inpatient_providers)
        * 100,
        4
    ),
    "%"
)

# --------------------------------------------------
# CHECK PROVIDER FEATURES
# --------------------------------------------------

provider_feature_columns = [
    c for c in provider.columns
    if c != "PRVDR_NUM"
]

print("\nProvider feature columns:")
print(provider_feature_columns)

# --------------------------------------------------
# MERGE
# --------------------------------------------------

inpatient_enriched = inpatient.merge(
    provider,
    on="PRVDR_NUM",
    how="left",
    validate="many_to_one",
    suffixes=("", "_PROVIDER")
)

print("\n====================================")
print("INPATIENT + PROVIDER JOIN")
print("====================================")

print("Original inpatient shape:", inpatient.shape)
print("Provider feature shape:", provider.shape)
print("Merged shape:", inpatient_enriched.shape)

print("\nOriginal rows:", len(inpatient))
print("Merged rows:", len(inpatient_enriched))

print(
    "\nRow count preserved:",
    len(inpatient) == len(inpatient_enriched)
)

# --------------------------------------------------
# CLAIM KEY VALIDATION
# --------------------------------------------------

print(
    "\nOriginal unique CLAIM_KEY:",
    inpatient["CLAIM_KEY"].nunique()
)

print(
    "Merged unique CLAIM_KEY:",
    inpatient_enriched["CLAIM_KEY"].nunique()
)

print(
    "Duplicate CLAIM_KEY after merge:",
    inpatient_enriched["CLAIM_KEY"].duplicated().sum()
)

# --------------------------------------------------
# PROVIDER FEATURE MATCH
# --------------------------------------------------

has_provider_features = (
    inpatient_enriched[provider_feature_columns]
    .notna()
    .any(axis=1)
)

print(
    "\nClaims with provider features:",
    has_provider_features.sum()
)

print(
    "Claims without provider features:",
    (~has_provider_features).sum()
)

print(
    "Claim-level provider feature match rate:",
    round(
        has_provider_features.mean() * 100,
        4
    ),
    "%"
)

# --------------------------------------------------
# MISSING VALUES
# --------------------------------------------------

print("\n====================================")
print("PROVIDER FEATURE MISSING VALUES")
print("====================================")

print(
    inpatient_enriched[
        provider_feature_columns
    ].isna().sum()
)

# --------------------------------------------------
# SAVE
# --------------------------------------------------

output_path = (
    "../data/processed/primary/"
    "inpatient_claims_fully_enriched.csv"
)

inpatient_enriched.to_csv(
    output_path,
    index=False
)

print("\n====================================")
print("SAVED")
print("====================================")

print("Saved:", output_path)
print("Final shape:", inpatient_enriched.shape)

INPATIENT + PROVIDER PRE-CHECK
Inpatient shape: (66773, 74)
Provider feature shape: (2675, 19)

Inpatient providers: 2675
Provider feature records: 2675

PROVIDER KEY CHECK
Duplicate PRVDR_NUM in provider features: 0
Unique provider feature keys: 2675

PROVIDER ID OVERLAP
Unique inpatient providers: 2675
Provider feature IDs: 2675
Matched providers: 2675
Unmatched inpatient providers: 0
Provider match rate: 100.0 %

Provider feature columns:
['CLAIM_COUNT', 'UNIQUE_CLAIM_COUNT', 'TOTAL_PAYMENT', 'AVG_PAYMENT', 'MEDIAN_PAYMENT', 'MAX_PAYMENT', 'AVG_CLAIM_DURATION', 'MAX_CLAIM_DURATION', 'AVG_DIAGNOSIS_COUNT', 'AVG_PROCEDURE_COUNT', 'NEGATIVE_PAYMENT_COUNT', 'PRIMARY_PAYER_PAYMENT_COUNT', 'NEGATIVE_PAYMENT_RATE', 'PRIMARY_PAYER_PAYMENT_RATE', 'UNIQUE_BENEFICIARIES', 'CLAIMS_PER_BENEFICIARY', 'PAYMENT_STD', 'CLAIM_DURATION_STD']

INPATIENT + PROVIDER JOIN
Original inpatient shape: (66773, 74)
Provider feature shape: (2675, 19)
Merged shape: (66773, 92)

Original rows: 66773
Merged rows: 6

In [5]:
import pandas as pd

carrier = pd.read_csv(
    "../data/processed/primary/Carrier_Claims_Features.csv",
    low_memory=False
)

beneficiary = pd.read_csv(
    "../data/processed/primary/beneficiary_longitudinal.csv",
    low_memory=False
)

print("====================================")
print("CARRIER + BENEFICIARY PRE-CHECK")
print("====================================")

print("Carrier shape:", carrier.shape)
print("Beneficiary shape:", beneficiary.shape)

print("\nCarrier columns:")
print(carrier.columns.tolist())

# -------------------------------
# STANDARDIZE YEAR
# -------------------------------

carrier["claim_year"] = pd.to_numeric(
    carrier["claim_year"],
    errors="coerce"
).astype("Int64")

beneficiary = beneficiary.rename(
    columns={"YEAR": "claim_year"}
)

beneficiary["claim_year"] = pd.to_numeric(
    beneficiary["claim_year"],
    errors="coerce"
).astype("Int64")

# -------------------------------
# BASIC CLAIM CHECKS
# -------------------------------

print("\n====================================")
print("CARRIER CLAIM CHECK")
print("====================================")

print("Carrier rows:", len(carrier))
print(
    "Unique CLM_ID:",
    carrier["CLM_ID"].nunique()
)

print(
    "Duplicate CLM_ID:",
    carrier["CLM_ID"].duplicated().sum()
)

print(
    "Unique beneficiaries:",
    carrier["DESYNPUF_ID"].nunique()
)

print("\nCarrier claim years:")
print(
    carrier["claim_year"]
    .value_counts(dropna=False)
    .sort_index()
)

# -------------------------------
# BENEFICIARY KEY CHECK
# -------------------------------

print("\n====================================")
print("BENEFICIARY-YEAR KEY CHECK")
print("====================================")

duplicate_bene_year = beneficiary.duplicated(
    ["DESYNPUF_ID", "claim_year"]
).sum()

print(
    "Duplicate beneficiary + year rows:",
    duplicate_bene_year
)

# -------------------------------
# BENEFICIARY ID OVERLAP
# -------------------------------

carrier_ids = set(
    carrier["DESYNPUF_ID"].dropna()
)

beneficiary_ids = set(
    beneficiary["DESYNPUF_ID"].dropna()
)

matched_ids = carrier_ids & beneficiary_ids

print("\n====================================")
print("BENEFICIARY ID OVERLAP")
print("====================================")

print(
    "Carrier unique beneficiaries:",
    len(carrier_ids)
)

print(
    "Beneficiary unique IDs:",
    len(beneficiary_ids)
)

print(
    "Matched beneficiary IDs:",
    len(matched_ids)
)

print(
    "Unmatched carrier beneficiary IDs:",
    len(carrier_ids - beneficiary_ids)
)

print(
    "ID match rate:",
    round(
        len(matched_ids) / len(carrier_ids) * 100,
        4
    ),
    "%"
)

# -------------------------------
# BENEFICIARY-YEAR MATCH
# -------------------------------

beneficiary_keys = set(
    zip(
        beneficiary["DESYNPUF_ID"],
        beneficiary["claim_year"]
    )
)

carrier_keys = zip(
    carrier["DESYNPUF_ID"],
    carrier["claim_year"]
)

matched_rows = sum(
    key in beneficiary_keys
    for key in carrier_keys
)

print("\n====================================")
print("BENEFICIARY-YEAR MATCHING")
print("====================================")

print(
    "Total carrier rows:",
    len(carrier)
)

print(
    "Matched beneficiary-year rows:",
    matched_rows
)

print(
    "Unmatched beneficiary-year rows:",
    len(carrier) - matched_rows
)

print(
    "Match rate:",
    round(
        matched_rows / len(carrier) * 100,
        4
    ),
    "%"
)

# -------------------------------
# MERGE
# -------------------------------

carrier_beneficiary = carrier.merge(
    beneficiary,
    on=["DESYNPUF_ID", "claim_year"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_BENE")
)

print("\n====================================")
print("CARRIER + BENEFICIARY JOIN")
print("====================================")

print(
    "Original carrier shape:",
    carrier.shape
)

print(
    "Beneficiary shape:",
    beneficiary.shape
)

print(
    "Merged shape:",
    carrier_beneficiary.shape
)

print(
    "\nOriginal rows:",
    len(carrier)
)

print(
    "Merged rows:",
    len(carrier_beneficiary)
)

print(
    "Row count preserved:",
    len(carrier) == len(carrier_beneficiary)
)

# -------------------------------
# CLAIM VALIDATION
# -------------------------------

print(
    "\nOriginal unique CLM_ID:",
    carrier["CLM_ID"].nunique()
)

print(
    "Merged unique CLM_ID:",
    carrier_beneficiary["CLM_ID"].nunique()
)

print(
    "Duplicate CLM_ID after merge:",
    carrier_beneficiary["CLM_ID"].duplicated().sum()
)

# -------------------------------
# BENEFICIARY FEATURE MATCH
# -------------------------------

beneficiary_feature_cols = [
    c for c in beneficiary.columns
    if c not in ["DESYNPUF_ID", "claim_year"]
]

has_beneficiary_data = (
    carrier_beneficiary[beneficiary_feature_cols]
    .notna()
    .any(axis=1)
)

print(
    "\nClaims with beneficiary-year data:",
    has_beneficiary_data.sum()
)

print(
    "Claims without beneficiary-year data:",
    (~has_beneficiary_data).sum()
)

print(
    "Actual match rate:",
    round(
        has_beneficiary_data.mean() * 100,
        4
    ),
    "%"
)

# -------------------------------
# SAVE
# -------------------------------

output_path = (
    "../data/processed/primary/"
    "carrier_claims_with_beneficiary_features.csv"
)

carrier_beneficiary.to_csv(
    output_path,
    index=False
)

print("\n====================================")
print("SAVED")
print("====================================")

print("Saved:", output_path)
print(
    "Final shape:",
    carrier_beneficiary.shape
)

CARRIER + BENEFICIARY PRE-CHECK
Carrier shape: (4741335, 21)
Beneficiary shape: (343644, 33)

Carrier columns:
['CLM_ID', 'DESYNPUF_ID', 'total_claim_payment_amt', 'total_allowed_charge_amt', 'total_deductible_amt', 'total_coinsurance_amt', 'total_primary_payer_paid_amt', 'avg_payment_per_line', 'payment_to_allowed_ratio', 'line_count', 'unique_hcpcs_count', 'max_line_payment', 'diagnosis_count', 'unique_diagnosis_count', 'primary_provider_npi', 'distinct_provider_count_on_claim', 'provider_claim_volume', 'provider_avg_claim_payment', 'claim_year', 'claim_month', 'claim_day_of_week']

CARRIER CLAIM CHECK
Carrier rows: 4741335
Unique CLM_ID: 4741335
Duplicate CLM_ID: 0
Unique beneficiaries: 98626

Carrier claim years:
claim_year
2008    1715402
2009    1862973
2010    1162960
Name: count, dtype: Int64

BENEFICIARY-YEAR KEY CHECK
Duplicate beneficiary + year rows: 0

BENEFICIARY ID OVERLAP
Carrier unique beneficiaries: 98626
Beneficiary unique IDs: 116352
Matched beneficiary IDs: 98626
U

In [6]:
import pandas as pd

paths = {
    "OUTPATIENT": "../data/processed/primary/outpatient_claims_with_beneficiary_features.csv",
    "INPATIENT": "../data/processed/primary/inpatient_claims_fully_enriched.csv",
    "CARRIER": "../data/processed/primary/carrier_claims_with_beneficiary_features.csv"
}

datasets = {}

for name, path in paths.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    df = pd.read_csv(
        path,
        nrows=5,
        low_memory=False
    )

    datasets[name] = df

    print("Shape information:")
    print("Columns:", len(df.columns))

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

# --------------------------------------------------
# COLUMN COMPARISON
# --------------------------------------------------

column_sets = {
    name: set(df.columns)
    for name, df in datasets.items()
}

print("\n" + "=" * 70)
print("COLUMN OVERLAP")
print("=" * 70)

for name, cols in column_sets.items():
    print(f"{name}: {len(cols)} columns")

common_all = (
    column_sets["OUTPATIENT"]
    & column_sets["INPATIENT"]
    & column_sets["CARRIER"]
)

print("\nColumns common to ALL THREE:")
print(sorted(common_all))

print("\nNumber common to all three:", len(common_all))

# Pairwise overlaps
print("\nOUTPATIENT ∩ INPATIENT:")
print(sorted(
    column_sets["OUTPATIENT"]
    & column_sets["INPATIENT"]
))

print("\nOUTPATIENT ∩ CARRIER:")
print(sorted(
    column_sets["OUTPATIENT"]
    & column_sets["CARRIER"]
))

print("\nINPATIENT ∩ CARRIER:")
print(sorted(
    column_sets["INPATIENT"]
    & column_sets["CARRIER"]
))

# Source-specific columns
print("\n" + "=" * 70)
print("SOURCE-SPECIFIC COLUMNS")
print("=" * 70)

for name, cols in column_sets.items():

    other_cols = set()

    for other_name, other_set in column_sets.items():
        if other_name != name:
            other_cols |= other_set

    specific = sorted(cols - other_cols)

    print(f"\n{name} SOURCE-SPECIFIC:")
    print(specific)
    print("Count:", len(specific))


OUTPATIENT
Shape information:
Columns: 57

Columns:
['CLAIM_KEY', 'DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT', 'TOTAL_REIMBURSEMENT', 'CLM_FROM_DT', 'CLM_THRU_DT', 'CLAIM_DURATION_DAYS', 'CLAIM_YEAR', 'CLAIM_MONTH', 'DIAGNOSIS_COUNT', 'PROCEDURE_COUNT', 'HCPCS_COUNT', 'HAS_DIAGNOSIS', 'HAS_PROCEDURE', 'HAS_HCPCS', 'HAS_NEGATIVE_PAYMENT', 'HAS_PRIMARY_PAYER_PAYMENT', 'IS_SEGMENT_2', 'HAS_SEGMENT_1_MATCH', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_O